In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, precision_recall_curve
)
from catboost import CatBoostClassifier
import lightgbm as lgb

warnings.filterwarnings('ignore')

INPUT_PATH = '~/Downloads/files/features.csv'
RANDOM_STATE = 42
N_SPLITS = 5
ITER_GRID = [100, 200, 300, 400, 600, 800, 1200]
PLATEAU_TOL = 0.005

ID_COLUMNS = ['regnum', 'notif_num', 'number', 'notification_purchase_number',
              'purchase_code', 'customer_spz_code', 'inn', 'customer_inn',
              'customer_kpp', 'kpp', 'ogrn', 'customer_okpo',
              'suppliers_0_inn', 'suppliers_0_kpp', 'suppliers_1_inn', 'suppliers_1_kpp',
              'suppliers_2_inn', 'suppliers_2_kpp', 'suppliers_3_inn', 'suppliers_3_kpp']

df = pd.read_csv(INPUT_PATH, dtype={c: str for c in ID_COLUMNS}, low_memory=False)
df['sign_date'] = pd.to_datetime(df['sign_date'])
df = df.sort_values(['sign_date', 'eis_url']).reset_index(drop=True)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")

id_and_leak_cols = [
    'inn', 'customer_inn', 'customer_kpp', 'kpp', 'ogrn',
    'regnum', 'number', 'notif_num', 'purchase_code', 'notification_purchase_number',
    'customer_spz_code', 'customer_okpo',
    'suppliers_0_inn', 'suppliers_0_kpp', 'suppliers_1_inn', 'suppliers_1_kpp',
    'suppliers_2_inn', 'suppliers_2_kpp', 'suppliers_3_inn', 'suppliers_3_kpp',
    'supplier_inns', 'supplier_kpps',
    'customer_full_name', 'customer_name', 'customer_okopf_name',
    'customer_subordination_type_name',
    'suppliers_0_full_name', 'suppliers_1_full_name', 'suppliers_2_full_name',
    'suppliers_3_full_name', 'supplier_names',
    'full_name', 'short_name', 'opf_name', 'single_supplier_reason_name',
    'contract_region_name', 'customer_region_name', 'supplier_region_name',
    'eis_url', 'notification_eis_url',
    'contract_subject', 'products', 'product_codes', 'product_names', 'foundation',
    'sign_date', 'execution_start_date', 'execution_end_date',
    'placement_date', 'publish_date',
    'price_original_currency',
    'version_number', 'currency', 'neg_amount',
    'has_nmck',
    'collusion',
]

categorical_features = [
    'fz', 'placing_way', 'single_supplier_reason_code',
    'customer_okopf_code', 'customer_subordination_type_code', 'opf_code',
    'okpd2_section', 'fed_district_code',
    'contract_region_code', 'customer_region_code', 'supplier_region_code',
    'industry', 'subindustry',
    'sign_year', 'sign_month', 'sign_quarter', 'sign_dayofweek',
]

feature_cols = [c for c in df.columns if c not in id_and_leak_cols]
categorical_features = [c for c in categorical_features if c in feature_cols]

print(f"Features: {len(feature_cols)} total, {len(categorical_features)} categorical")

y = df['collusion'].astype(int).values
groups = df['inn'].values

X = df[feature_cols].copy()
for c in categorical_features:
    X[c] = X[c].astype('object').fillna('NA').astype(str)
numeric_features = [c for c in feature_cols if c not in categorical_features]
for c in numeric_features:
    X[c] = pd.to_numeric(X[c], errors='coerce')
cat_feature_indices = [X.columns.get_loc(c) for c in categorical_features]

print(f"X shape {X.shape}, collusion rate {y.mean():.3f}")

def evaluate_oof(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'pr_auc': average_precision_score(y_true, y_proba),
        'roc_auc': roc_auc_score(y_true, y_proba),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
    }

def best_f1_threshold(y_true, y_proba):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx = np.argmax(f1s[:-1])
    return thresholds[best_idx], f1s[best_idx]

def make_catboost(n_iters):
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return CatBoostClassifier(
        iterations=n_iters, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
        scale_pos_weight=pos_weight, random_seed=RANDOM_STATE,
        verbose=False, allow_writing_files=False)

def catboost_oof_groupkfold(n_iters):
    gkf = GroupKFold(n_splits=N_SPLITS)
    oof = np.zeros(len(y))
    for tr, te in gkf.split(X, y, groups=groups):
        m = make_catboost(n_iters)
        m.fit(X.iloc[tr], y[tr], cat_features=cat_feature_indices)
        oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return oof

print("Tuning number of iterations (GroupKFold PR-AUC)")
iter_curve = {}
for n in ITER_GRID:
    oof_n = catboost_oof_groupkfold(n)
    pr = average_precision_score(y, oof_n)
    iter_curve[n] = float(pr)
    print(f"iterations {n}: PR-AUC {pr:.4f}")

max_pr = max(iter_curve.values())
CHOSEN_ITERATIONS = min(n for n, pr in iter_curve.items() if pr >= max_pr - PLATEAU_TOL)
print(f"Max PR-AUC {max_pr:.4f}, chosen iterations {CHOSEN_ITERATIONS}")

results = {}

print(f"CatBoost StratifiedKFold, iterations {CHOSEN_ITERATIONS}")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_proba_skf = np.zeros(len(y))
for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y)):
    model = make_catboost(CHOSEN_ITERATIONS)
    model.fit(X.iloc[tr_idx], y[tr_idx], cat_features=cat_feature_indices)
    oof_proba_skf[te_idx] = model.predict_proba(X.iloc[te_idx])[:, 1]
    print(f"fold {fold+1}/{N_SPLITS} done")
results['catboost_stratified'] = evaluate_oof(y, oof_proba_skf)
thr_skf, f1_skf = best_f1_threshold(y, oof_proba_skf)
results['catboost_stratified']['best_f1'] = f1_skf
results['catboost_stratified']['best_threshold'] = float(thr_skf)

print("CatBoost GroupKFold by inn")
gkf = GroupKFold(n_splits=N_SPLITS)
oof_proba_gkf = np.zeros(len(y))
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups)):
    model = make_catboost(CHOSEN_ITERATIONS)
    model.fit(X.iloc[tr_idx], y[tr_idx], cat_features=cat_feature_indices)
    oof_proba_gkf[te_idx] = model.predict_proba(X.iloc[te_idx])[:, 1]
    print(f"fold {fold+1}/{N_SPLITS} done, test suppliers {len(np.unique(groups[te_idx]))}")
results['catboost_group'] = evaluate_oof(y, oof_proba_gkf)
thr_gkf, f1_gkf = best_f1_threshold(y, oof_proba_gkf)
results['catboost_group']['best_f1'] = f1_gkf
results['catboost_group']['best_threshold'] = float(thr_gkf)

X_lgb = df[feature_cols].copy()
for c in categorical_features:
    X_lgb[c] = X_lgb[c].astype('category')
for c in numeric_features:
    X_lgb[c] = pd.to_numeric(X_lgb[c], errors='coerce')

def make_lgbm(n_iters):
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return lgb.LGBMClassifier(
        n_estimators=n_iters, learning_rate=0.05, num_leaves=31, max_depth=6,
        reg_lambda=3.0, scale_pos_weight=pos_weight,
        random_state=RANDOM_STATE, verbose=-1)

print(f"LightGBM, n_estimators {CHOSEN_ITERATIONS}")
oof_lgb_skf = np.zeros(len(y))
for fold, (tr_idx, te_idx) in enumerate(skf.split(X_lgb, y)):
    model = make_lgbm(CHOSEN_ITERATIONS)
    model.fit(X_lgb.iloc[tr_idx], y[tr_idx], categorical_feature=categorical_features)
    oof_lgb_skf[te_idx] = model.predict_proba(X_lgb.iloc[te_idx])[:, 1]
    print(f"stratified fold {fold+1}/{N_SPLITS} done")
results['lightgbm_stratified'] = evaluate_oof(y, oof_lgb_skf)
_, f1 = best_f1_threshold(y, oof_lgb_skf)
results['lightgbm_stratified']['best_f1'] = f1

oof_lgb_gkf = np.zeros(len(y))
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_lgb, y, groups=groups)):
    model = make_lgbm(CHOSEN_ITERATIONS)
    model.fit(X_lgb.iloc[tr_idx], y[tr_idx], categorical_feature=categorical_features)
    oof_lgb_gkf[te_idx] = model.predict_proba(X_lgb.iloc[te_idx])[:, 1]
    print(f"group fold {fold+1}/{N_SPLITS} done")
results['lightgbm_group'] = evaluate_oof(y, oof_lgb_gkf)
_, f1 = best_f1_threshold(y, oof_lgb_gkf)
results['lightgbm_group']['best_f1'] = f1

print("Metrics comparison")
results_df = pd.DataFrame(results).T
results_df = results_df[['pr_auc', 'roc_auc', 'f1', 'best_f1', 'precision', 'recall']]
print(results_df.round(4).to_string())

baseline_pr = y.mean()
print(f"Baseline PR-AUC (positive share) {baseline_pr:.4f}")

print("Feature importance (CatBoost, full data)")
model_full = make_catboost(CHOSEN_ITERATIONS)
model_full.fit(X, y, cat_features=cat_feature_indices)
importances = model_full.get_feature_importance()
fi_df = pd.DataFrame({'feature': X.columns, 'importance': importances}) \
    .sort_values('importance', ascending=False).reset_index(drop=True)
print(fi_df.head(25).to_string())


Loaded 18537 rows, 116 columns
Features: 61 total, 17 categorical
X shape (18537, 61), collusion rate 0.079
Tuning number of iterations (GroupKFold PR-AUC)
iterations 100: PR-AUC 0.4158
iterations 200: PR-AUC 0.5311
iterations 300: PR-AUC 0.5311
iterations 400: PR-AUC 0.5309
iterations 600: PR-AUC 0.5319
iterations 800: PR-AUC 0.5346
iterations 1200: PR-AUC 0.5352
Max PR-AUC 0.5352, chosen iterations 200
CatBoost StratifiedKFold, iterations 200
fold 1/5 done
fold 2/5 done
fold 3/5 done
fold 4/5 done
fold 5/5 done
CatBoost GroupKFold by inn
fold 1/5 done, test suppliers 37
fold 2/5 done, test suppliers 37
fold 3/5 done, test suppliers 37
fold 4/5 done, test suppliers 37
fold 5/5 done, test suppliers 36
LightGBM, n_estimators 200
stratified fold 1/5 done
stratified fold 2/5 done
stratified fold 3/5 done
stratified fold 4/5 done
stratified fold 5/5 done
group fold 1/5 done
group fold 2/5 done
group fold 3/5 done
group fold 4/5 done
group fold 5/5 done
Metrics comparison
                  

In [2]:
import warnings

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_score, recall_score, precision_recall_curve)

warnings.filterwarnings('ignore')

INPUT_PATH = '~/Downloads/files/features.csv'
RANDOM_STATE = 42
N_SPLITS = 5
N_ITERATIONS = 200
USE_SIGN_YEAR = False

ID_COLUMNS = ['regnum', 'notif_num', 'number', 'notification_purchase_number',
              'purchase_code', 'customer_spz_code', 'inn', 'customer_inn',
              'customer_kpp', 'kpp', 'ogrn', 'customer_okpo',
              'suppliers_0_inn', 'suppliers_0_kpp', 'suppliers_1_inn', 'suppliers_1_kpp',
              'suppliers_2_inn', 'suppliers_2_kpp', 'suppliers_3_inn', 'suppliers_3_kpp']

ID_AND_LEAK = set([
    'inn', 'customer_inn', 'customer_kpp', 'kpp', 'ogrn', 'regnum', 'number', 'notif_num',
    'purchase_code', 'notification_purchase_number', 'customer_spz_code', 'customer_okpo',
    'suppliers_0_inn', 'suppliers_0_kpp', 'suppliers_1_inn', 'suppliers_1_kpp',
    'suppliers_2_inn', 'suppliers_2_kpp', 'suppliers_3_inn', 'suppliers_3_kpp',
    'supplier_inns', 'supplier_kpps', 'customer_full_name', 'customer_name',
    'customer_okopf_name', 'customer_subordination_type_name', 'suppliers_0_full_name',
    'suppliers_1_full_name', 'suppliers_2_full_name', 'suppliers_3_full_name',
    'supplier_names', 'full_name', 'short_name', 'opf_name', 'single_supplier_reason_name',
    'contract_region_name', 'customer_region_name', 'supplier_region_name',
    'eis_url', 'notification_eis_url', 'contract_subject', 'products', 'product_codes',
    'product_names', 'foundation', 'sign_date', 'execution_start_date', 'execution_end_date',
    'placement_date', 'publish_date', 'price_original_currency', 'version_number', 'currency',
    'neg_amount', 'collusion', 'has_nmck'
])

DROP_ALWAYS = ['industry', 'subindustry', 'days_sign_to_placement']
if not USE_SIGN_YEAR:
    DROP_ALWAYS = DROP_ALWAYS + ['sign_year']

NMCK_FEATURES = ['notification_max_price', 'price_drop_pct', 'price_drop_abs',
                 'price_drop_vs_supplier_avg']

E_REQUIRED = ['notification_max_price', 'placing_way', 'okpd2_section',
              'customer_okopf_code', 'opf_code', 'contract_duration_days',
              'days_sign_to_exec_start']

CATEGORICAL_ALL = ['fz', 'placing_way', 'single_supplier_reason_code', 'customer_okopf_code',
    'customer_subordination_type_code', 'opf_code', 'okpd2_section', 'fed_district_code',
    'contract_region_code', 'customer_region_code', 'supplier_region_code',
    'sign_year', 'sign_month', 'sign_quarter', 'sign_dayofweek']

STRATEGIES = {
    'A': 'All 44+223, with reserve price',
    'B': 'All 44+223, no reserve price',
    'C': '44-FZ only, with reserve price',
    'D': '44-FZ only, no reserve price',
    'E': '44-FZ only, complete rows',
}

def load_data():
    df = pd.read_csv(INPUT_PATH, dtype={c: str for c in ID_COLUMNS}, low_memory=False)
    df['sign_date'] = pd.to_datetime(df['sign_date'])
    df = df.sort_values(['sign_date', 'eis_url']).reset_index(drop=True)
    return df

def build_strategy(df, name):
    d = df.copy()
    if name in ('C', 'D', 'E'):
        d = d[d['fz'].astype(str) == '44'].reset_index(drop=True)
    if name == 'E':
        mask = d[E_REQUIRED].notna().all(axis=1)
        d = d[mask].reset_index(drop=True)
    feats = [c for c in d.columns if c not in ID_AND_LEAK]
    feats = [c for c in feats if c not in DROP_ALWAYS]
    if name in ('B', 'D'):
        feats = [c for c in feats if c not in NMCK_FEATURES]
    return d, feats

def prep_X(d, feats):
    cats = [c for c in CATEGORICAL_ALL if c in feats]
    X = d[feats].copy()
    for c in cats:
        X[c] = X[c].astype('object').fillna('NA').astype(str)
    for c in [f for f in feats if f not in cats]:
        X[c] = pd.to_numeric(X[c], errors='coerce')
    cat_idx = [X.columns.get_loc(c) for c in cats]
    return X, cat_idx

def make_model(pos_weight):
    return CatBoostClassifier(
        iterations=N_ITERATIONS, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
        scale_pos_weight=pos_weight, random_seed=RANDOM_STATE,
        verbose=False, allow_writing_files=False)

def compute_metrics(y_true, proba):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    best_idx = np.argmax(f1s[:-1])
    best_thr = thr[best_idx]
    y_pred = (proba >= best_thr).astype(int)
    return {
        'pr_auc': float(average_precision_score(y_true, proba)),
        'roc_auc': float(roc_auc_score(y_true, proba)),
        'best_f1': float(f1s[:-1].max()),
        'best_threshold': float(best_thr),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'baseline_pr': float(y_true.mean()),
        'n': int(len(y_true)),
        'n_pos': int(y_true.sum()),
    }

def eval_groupkfold(d, feats):
    X, cat_idx = prep_X(d, feats)
    y = d['collusion'].astype(int).values
    groups = d['inn'].values
    pos_w = (y == 0).sum() / (y == 1).sum()
    gkf = GroupKFold(n_splits=N_SPLITS)
    oof = np.zeros(len(y))
    for tr, te in gkf.split(X, y, groups):
        m = make_model(pos_w)
        m.fit(X.iloc[tr], y[tr], cat_features=cat_idx)
        oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return compute_metrics(y, oof), oof

def eval_timesplit(d, feats):
    feats_t = [c for c in feats if c != 'sign_year']
    X, cat_idx = prep_X(d, feats_t)
    y = d['collusion'].astype(int).values
    year = d['sign_date'].dt.year.values
    tr = (year >= 2014) & (year <= 2022)
    te = (year >= 2023) & (year <= 2024)
    pos_w = (y[tr] == 0).sum() / (y[tr] == 1).sum()
    m = make_model(pos_w)
    m.fit(X[tr], y[tr], cat_features=cat_idx)
    proba = m.predict_proba(X[te])[:, 1]
    return compute_metrics(y[te], proba)

def signyear_ablation(df):
    d = df.copy()
    d = d[d['fz'].astype(str) == '44'].reset_index(drop=True)
    d = d[d[E_REQUIRED].notna().all(axis=1)].reset_index(drop=True)
    base_feats = [c for c in d.columns if c not in ID_AND_LEAK
                  and c not in ['industry', 'subindustry', 'days_sign_to_placement']]
    feats_with = base_feats if 'sign_year' in base_feats else base_feats + ['sign_year']
    feats_with = [c for c in feats_with if c in d.columns]
    feats_no = [c for c in feats_with if c != 'sign_year']
    print("sign_year ablation (strategy E, GroupKFold)")
    m_with, _ = eval_groupkfold(d, feats_with)
    m_without, _ = eval_groupkfold(d, feats_no)
    delta = m_with['pr_auc'] - m_without['pr_auc']
    print(f"with sign_year: PR-AUC {m_with['pr_auc']:.4f} ROC-AUC {m_with['roc_auc']:.4f} F1 {m_with['best_f1']:.4f}")
    print(f"without sign_year: PR-AUC {m_without['pr_auc']:.4f} ROC-AUC {m_without['roc_auc']:.4f} F1 {m_without['best_f1']:.4f}")
    print(f"sign_year contribution to PR-AUC {delta:+.4f}")
    return {'with_sign_year': m_with, 'without_sign_year': m_without, 'delta_pr_auc': float(delta)}

def print_summary(results):
    rows = []
    for name in STRATEGIES:
        r = results[name]
        gk = r['groupkfold']
        ts = r['timesplit']
        rows.append({
            'Strat': name, 'Description': r['desc'], 'Rows': r['n'], 'Cartels': r['n_pos'],
            'Features': r['n_features'],
            'GKF PR-AUC': round(gk['pr_auc'], 4), 'GKF ROC': round(gk['roc_auc'], 4),
            'GKF F1': round(gk['best_f1'], 4),
            'GKF over base': f"{gk['pr_auc'] / gk['baseline_pr']:.1f}x",
            'Time PR-AUC': round(ts['pr_auc'], 4),
            'Time ROC': round(ts['roc_auc'], 4),
        })
    summary = pd.DataFrame(rows)
    print(f"Strategy comparison (CatBoost, iterations {N_ITERATIONS})")
    print(summary.to_string(index=False))

def main():
    df = load_data()
    print(f"Loaded {len(df)} rows. N_ITERATIONS {N_ITERATIONS}, USE_SIGN_YEAR {USE_SIGN_YEAR}")

    results = {}
    for name in list(STRATEGIES.keys()):
        desc = STRATEGIES[name]
        d, feats = build_strategy(df, name)
        print(f"Strategy {name}: {desc}")
        print(f"rows {len(d)}, cartels {int(d['collusion'].sum())} ({d['collusion'].mean() * 100:.1f}%), features {len(feats)}")
        gkf_m, oof = eval_groupkfold(d, feats)
        ts_m = eval_timesplit(d, feats)
        print(f"GroupKFold: PR-AUC {gkf_m['pr_auc']:.4f} ROC-AUC {gkf_m['roc_auc']:.4f} F1 {gkf_m['best_f1']:.4f} baseline {gkf_m['baseline_pr']:.4f}")
        print(f"Time-split: PR-AUC {ts_m['pr_auc']:.4f} ROC-AUC {ts_m['roc_auc']:.4f} F1 {ts_m['best_f1']:.4f} baseline {ts_m['baseline_pr']:.4f}")
        results[name] = {'desc': desc, 'n': len(d), 'n_pos': int(d['collusion'].sum()),
                         'n_features': len(feats), 'groupkfold': gkf_m, 'timesplit': ts_m}

    print_summary(results)
    signyear_ablation(df)

main()


Loaded 18537 rows. N_ITERATIONS 200, USE_SIGN_YEAR False
Strategy A: All 44+223, with reserve price
rows 18537, cartels 1471 (7.9%), features 57
GroupKFold: PR-AUC 0.4139 ROC-AUC 0.8620 F1 0.4226 baseline 0.0794
Time-split: PR-AUC 0.6577 ROC-AUC 0.9249 F1 0.6555 baseline 0.0988
Strategy B: All 44+223, no reserve price
rows 18537, cartels 1471 (7.9%), features 53
GroupKFold: PR-AUC 0.3773 ROC-AUC 0.8420 F1 0.4067 baseline 0.0794
Time-split: PR-AUC 0.6607 ROC-AUC 0.9348 F1 0.6627 baseline 0.0988
Strategy C: 44-FZ only, with reserve price
rows 16989, cartels 1452 (8.5%), features 57
GroupKFold: PR-AUC 0.3880 ROC-AUC 0.8435 F1 0.4193 baseline 0.0855
Time-split: PR-AUC 0.6805 ROC-AUC 0.9211 F1 0.6556 baseline 0.0988
Strategy D: 44-FZ only, no reserve price
rows 16989, cartels 1452 (8.5%), features 53
GroupKFold: PR-AUC 0.3606 ROC-AUC 0.8125 F1 0.3589 baseline 0.0855
Time-split: PR-AUC 0.6816 ROC-AUC 0.9270 F1 0.6562 baseline 0.0988
Strategy E: 44-FZ only, complete rows
rows 13176, cartels 1

In [3]:
import warnings

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_score, recall_score, precision_recall_curve)
from cleanlab.filter import find_label_issues
from cleanlab.rank import get_label_quality_scores

warnings.filterwarnings('ignore')

INPUT_PATH = '~/Downloads/files/features.csv'
RANDOM_STATE = 42
N_SPLITS = 5
N_ITERATIONS = 200
USE_SIGN_YEAR = False
N_BOOTSTRAP = 1000
FILTER_METHODS = ['prune_by_noise_rate', 'prune_by_class', 'both']
FRAC_NOISE_GRID = [0.25, 0.5, 0.75, 1.0]

ID_COLUMNS = ['regnum','notif_num','number','notification_purchase_number','purchase_code',
              'customer_spz_code','inn','customer_inn','customer_kpp','kpp','ogrn','customer_okpo',
              'suppliers_0_inn','suppliers_0_kpp','suppliers_1_inn','suppliers_1_kpp',
              'suppliers_2_inn','suppliers_2_kpp','suppliers_3_inn','suppliers_3_kpp']

ID_AND_LEAK = set([
    'inn','customer_inn','customer_kpp','kpp','ogrn','regnum','number','notif_num',
    'purchase_code','notification_purchase_number','customer_spz_code','customer_okpo',
    'suppliers_0_inn','suppliers_0_kpp','suppliers_1_inn','suppliers_1_kpp',
    'suppliers_2_inn','suppliers_2_kpp','suppliers_3_inn','suppliers_3_kpp',
    'supplier_inns','supplier_kpps','customer_full_name','customer_name',
    'customer_okopf_name','customer_subordination_type_name','suppliers_0_full_name',
    'suppliers_1_full_name','suppliers_2_full_name','suppliers_3_full_name',
    'supplier_names','full_name','short_name','opf_name','single_supplier_reason_name',
    'contract_region_name','customer_region_name','supplier_region_name',
    'eis_url','notification_eis_url','contract_subject','products','product_codes',
    'product_names','foundation','sign_date','execution_start_date','execution_end_date',
    'placement_date','publish_date','price_original_currency','version_number','currency',
    'neg_amount','collusion','has_nmck'])

DROP_ALWAYS = ['industry','subindustry','days_sign_to_placement']
if not USE_SIGN_YEAR:
    DROP_ALWAYS = DROP_ALWAYS + ['sign_year']

E_REQUIRED = ['notification_max_price','placing_way','okpd2_section','customer_okopf_code',
              'opf_code','contract_duration_days','days_sign_to_exec_start']

CATEGORICAL_ALL = ['fz','placing_way','single_supplier_reason_code','customer_okopf_code',
    'customer_subordination_type_code','opf_code','okpd2_section','fed_district_code',
    'contract_region_code','customer_region_code','supplier_region_code',
    'sign_year','sign_month','sign_quarter','sign_dayofweek']

def load_strategy_E():
    df = pd.read_csv(INPUT_PATH, dtype={c: str for c in ID_COLUMNS}, low_memory=False)
    df['sign_date'] = pd.to_datetime(df['sign_date'])
    df = df.sort_values(['sign_date', 'eis_url']).reset_index(drop=True)
    df = df[df['fz'].astype(str) == '44'].reset_index(drop=True)
    mask = df[E_REQUIRED].notna().all(axis=1)
    df = df[mask].reset_index(drop=True)
    feats = [c for c in df.columns if c not in ID_AND_LEAK and c not in DROP_ALWAYS]
    return df, feats

def prep_X(df, feats):
    cats = [c for c in CATEGORICAL_ALL if c in feats]
    X = df[feats].copy()
    for c in cats:
        X[c] = X[c].astype('object').fillna('NA').astype(str)
    for c in [f for f in feats if f not in cats]:
        X[c] = pd.to_numeric(X[c], errors='coerce')
    cat_idx = [X.columns.get_loc(c) for c in cats]
    return X, cat_idx

def make_model(pos_weight):
    return CatBoostClassifier(
        iterations=N_ITERATIONS, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
        scale_pos_weight=pos_weight, random_seed=RANDOM_STATE,
        verbose=False, allow_writing_files=False)

def compute_metrics(y_true, proba):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1s = 2*prec*rec/(prec+rec+1e-9)
    bi = np.argmax(f1s[:-1])
    bt = thr[bi]
    yp = (proba >= bt).astype(int)
    return {
        'pr_auc': float(average_precision_score(y_true, proba)),
        'roc_auc': float(roc_auc_score(y_true, proba)),
        'best_f1': float(f1s[:-1].max()),
        'precision': float(precision_score(y_true, yp, zero_division=0)),
        'recall': float(recall_score(y_true, yp, zero_division=0)),
        'baseline_pr': float(y_true.mean()),
    }

def bootstrap_pr_ci(y_true, proba, n=N_BOOTSTRAP, alpha=0.05):
    rng = np.random.RandomState(RANDOM_STATE)
    N = len(y_true)
    vals = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        vals.append(average_precision_score(y_true[idx], proba[idx]))
    vals = np.array(vals)
    return float(np.percentile(vals, 100*alpha/2)), float(np.percentile(vals, 100*(1-alpha/2)))

def get_oof_probs(X, y, groups, cat_idx):
    pos_w = (y == 0).sum() / (y == 1).sum()
    gkf = GroupKFold(n_splits=N_SPLITS)
    oof = np.zeros(len(y))
    for tr, te in gkf.split(X, y, groups):
        m = make_model(pos_w)
        m.fit(X.iloc[tr], y[tr], cat_features=cat_idx)
        oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return np.column_stack([1 - oof, oof])

def find_issues(y, pred_probs, filter_by='prune_by_noise_rate', frac_noise=1.0):
    idx = find_label_issues(labels=y, pred_probs=pred_probs,
        return_indices_ranked_by='self_confidence',
        filter_by=filter_by, frac_noise=frac_noise)
    mask = np.zeros(len(y), dtype=bool)
    mask[idx] = True
    return mask

def apply_cl_per_fold(y_tr, issues_tr, strategy):
    y_new = y_tr.copy()
    keep = np.ones(len(y_tr), dtype=bool)
    if strategy == 'baseline':
        return y_new, keep
    if strategy == 'relabel_to_pos':
        y_new[issues_tr & (y_tr == 0)] = 1
    elif strategy == 'relabel_to_neg':
        y_new[issues_tr & (y_tr == 1)] = 0
    elif strategy == 'relabel_both':
        y_new[issues_tr] = 1 - y_tr[issues_tr]
    elif strategy == 'remove':
        keep = ~issues_tr
    return y_new, keep

def run_cl_strategy(X, y, groups, cat_idx, issues_mask, strategy, splits):
    oof = np.zeros(len(y))
    n_changed = 0
    for tr, te in splits:
        y_tr = y[tr]
        y_new, keep = apply_cl_per_fold(y_tr, issues_mask[tr], strategy)
        X_tr = X.iloc[tr][keep]
        y_fit = y_new[keep]
        n_changed += int((~keep).sum()) + int((y_new[keep] != y_tr[keep]).sum())
        pos_w = (y_fit == 0).sum() / (y_fit == 1).sum()
        m = make_model(pos_w)
        m.fit(X_tr, y_fit, cat_features=cat_idx)
        oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return oof, n_changed

def compute_global_issues(df, feats):
    X, cat_idx = prep_X(df, feats)
    y = df['collusion'].astype(int).values
    groups = df['inn'].values
    pred_probs = get_oof_probs(X, y, groups, cat_idx)
    issues_mask = find_issues(y, pred_probs)
    quality = get_label_quality_scores(labels=y, pred_probs=pred_probs)
    return X, y, groups, cat_idx, pred_probs, issues_mask, quality

def approach2_detect(df, pred_probs, issues_mask, quality):
    print("Approach 2: detecting suspicious contracts")
    out = df[['eis_url', 'inn', 'customer_inn', 'sign_date', 'price_rur',
              'price_drop_pct', 'collusion']].copy()
    out['oof_proba_collusion'] = pred_probs[:, 1]
    out['label_quality'] = quality
    out['is_label_issue'] = issues_mask
    missed = out[(out['collusion'] == False) & (out['is_label_issue'])].copy()
    missed = missed.sort_values('oof_proba_collusion', ascending=False)
    wrong = out[(out['collusion'] == True) & (out['is_label_issue'])].copy()
    wrong = wrong.sort_values('oof_proba_collusion', ascending=True)
    print(f"Total label issues {int(issues_mask.sum())}")
    print(f"Potentially missed cartels (False, predicted True) {len(missed)}")
    print(f"Potentially wrong accusations (True, predicted False) {len(wrong)}")
    missed.to_csv('missed_cartels.csv', index=False)
    wrong.to_csv('wrong_accusations.csv', index=False)
    print("Top 10 potentially missed cartels")
    cols = ['eis_url', 'price_rur', 'price_drop_pct', 'oof_proba_collusion']
    print(missed[cols].head(10).to_string(index=False))
    return {'n_issues': int(issues_mask.sum()), 'n_missed': len(missed), 'n_wrong': len(wrong)}

def approach1_quantitative(X, y, groups, cat_idx, issues_mask):
    print("Approach 1: impact of CL on model quality (GroupKFold)")
    strategies = ['baseline', 'relabel_to_pos', 'relabel_to_neg', 'relabel_both', 'remove']
    results = {}
    oof_store = {}
    gkf = GroupKFold(n_splits=N_SPLITS)
    splits = list(gkf.split(X, y, groups))
    for strat in strategies:
        oof, n_changed = run_cl_strategy(X, y, groups, cat_idx, issues_mask, strat, splits)
        m_res = compute_metrics(y, oof)
        lo, hi = bootstrap_pr_ci(y, oof)
        m_res['pr_auc_ci_low'] = lo
        m_res['pr_auc_ci_high'] = hi
        m_res['n_changed_approx'] = int(n_changed)
        results[strat] = m_res
        oof_store[strat] = oof
        print(f"{strat}: PR-AUC {m_res['pr_auc']:.4f} [95% CI {lo:.4f}; {hi:.4f}] "
              f"ROC-AUC {m_res['roc_auc']:.4f} F1 {m_res['best_f1']:.4f} changed {n_changed}")
    print("Significance over baseline (bootstrap)")
    rng = np.random.RandomState(RANDOM_STATE)
    N = len(y)
    base_oof = oof_store['baseline']
    boot_idx = [rng.randint(0, N, N) for _ in range(N_BOOTSTRAP)]
    for strat in strategies:
        if strat == 'baseline':
            continue
        deltas = []
        for idx in boot_idx:
            deltas.append(average_precision_score(y[idx], oof_store[strat][idx])
                          - average_precision_score(y[idx], base_oof[idx]))
        deltas = np.array(deltas)
        results[strat]['delta_vs_baseline_mean'] = float(deltas.mean())
        results[strat]['prob_better_than_baseline'] = float((deltas > 0).mean())
        print(f"{strat}: delta {deltas.mean():+.4f}, P(strategy > baseline) {(deltas>0).mean():.1%}")
    return results

def robustness_filter_by(X, y, groups, cat_idx, pred_probs):
    print("Robustness 1: issue-selection method (filter_by)")
    gkf = GroupKFold(n_splits=N_SPLITS)
    splits = list(gkf.split(X, y, groups))
    base_oof, _ = run_cl_strategy(X, y, groups, cat_idx, np.zeros(len(y), bool), 'baseline', splits)
    base_pr = average_precision_score(y, base_oof)
    print(f"baseline PR-AUC {base_pr:.4f}")
    for fb in FILTER_METHODS:
        mask = find_issues(y, pred_probs, filter_by=fb, frac_noise=1.0)
        for strat in ['remove', 'relabel_to_neg']:
            oof, nch = run_cl_strategy(X, y, groups, cat_idx, mask, strat, splits)
            pr = average_precision_score(y, oof)
            print(f"{fb} {strat}: PR-AUC {pr:.4f} (delta vs base {pr-base_pr:+.4f}, issues {int(mask.sum())})")

def robustness_frac_noise(X, y, groups, cat_idx, pred_probs):
    print("Robustness 2: fraction of pruned noise (frac_noise), strategy remove")
    gkf = GroupKFold(n_splits=N_SPLITS)
    splits = list(gkf.split(X, y, groups))
    base_oof, _ = run_cl_strategy(X, y, groups, cat_idx, np.zeros(len(y), bool), 'baseline', splits)
    base_pr = average_precision_score(y, base_oof)
    print(f"frac_noise 0.00 (baseline) PR-AUC {base_pr:.4f}")
    for fn in FRAC_NOISE_GRID:
        mask = find_issues(y, pred_probs, filter_by='prune_by_noise_rate', frac_noise=fn)
        oof, nch = run_cl_strategy(X, y, groups, cat_idx, mask, 'remove', splits)
        pr = average_precision_score(y, oof)
        print(f"frac_noise {fn:.2f} PR-AUC {pr:.4f} (delta vs base {pr-base_pr:+.4f}, removed {int(mask.sum())})")

def main():
    df, feats = load_strategy_E()
    print(f"Strategy E: {len(df)} rows, {len(feats)} features, "
          f"cartels {int(df['collusion'].sum())} ({df['collusion'].mean()*100:.1f}%). "
          f"N_ITERATIONS {N_ITERATIONS}, USE_SIGN_YEAR {USE_SIGN_YEAR}")

    X, y, groups, cat_idx, pred_probs, issues_mask, quality = compute_global_issues(df, feats)

    approach2_detect(df, pred_probs, issues_mask, quality)
    a1 = approach1_quantitative(X, y, groups, cat_idx, issues_mask)

    print("Summary (Approach 1: PR-AUC by CL strategy)")
    rows = []
    for strat, m in a1.items():
        rows.append({'CL strategy': strat, 'PR-AUC': round(m['pr_auc'], 4),
                     'CI_low': round(m.get('pr_auc_ci_low', 0), 4),
                     'CI_high': round(m.get('pr_auc_ci_high', 0), 4),
                     'P>base': (f"{m['prob_better_than_baseline']:.0%}"
                                if 'prob_better_than_baseline' in m else '-'),
                     'ROC-AUC': round(m['roc_auc'], 4), 'F1': round(m['best_f1'], 4),
                     'changed': m['n_changed_approx']})
    print(pd.DataFrame(rows).to_string(index=False))

    robustness_filter_by(X, y, groups, cat_idx, pred_probs)
    robustness_frac_noise(X, y, groups, cat_idx, pred_probs)

main()


Strategy E: 13176 rows, 57 features, cartels 1139 (8.6%). N_ITERATIONS 200, USE_SIGN_YEAR False
Approach 2: detecting suspicious contracts
Total label issues 1409
Potentially missed cartels (False, predicted True) 1235
Potentially wrong accusations (True, predicted False) 174
Top 10 potentially missed cartels
                                                                                           eis_url  price_rur  price_drop_pct  oof_proba_collusion
https://zakupki.gov.ru/epz/contract/contractCard/common-info.html?reestrNumber=2770701146524000033 8048875.25        0.000000             0.973839
https://zakupki.gov.ru/epz/contract/contractCard/common-info.html?reestrNumber=2770701146524000044  100990.00        0.000000             0.970517
https://zakupki.gov.ru/epz/contract/contractCard/common-info.html?reestrNumber=2420702215018000131 3322800.00        0.010011             0.957814
https://zakupki.gov.ru/epz/contract/contractCard/common-info.html?reestrNumber=2772000185024000161 29